## MITRE ATT&CK integration (Enterprise + ICS)

This notebook extracts attack techniques, mitigations and the relationships between them from MITRE ATT&CK, so they can be linked to the CISA exercise scenarios.

**What changed compared to the previous version:**
- **ICS ATT&CK added next to Enterprise ATT&CK.** One of the CISA manuals covers Industrial Control Systems. Enterprise ATT&CK describes IT attacks, while ICS ATT&CK describes attacks on industrial processes and their physical/business effects (e.g. *Loss of Safety*, *Loss of Productivity and Revenue*). Without it, the ICS scenarios cannot be mapped properly.
- **Mobile ATT&CK is deliberately not included.** None of the CISA scenarios involve attacks on mobile devices. The one phone-related event (MFA push notifications) is already covered by Enterprise ATT&CK (T1621).
- **A `domain` column was added** to every output table, so it is always clear which ATT&CK dataset a row comes from.
- **The ATT&CK files are downloaded automatically** from the team's fork of the `attack-stix-data` repository on GitHub, instead of being copied onto each computer by hand.
- **The code was reorganised into reusable functions** that run once per domain, instead of repeating the same steps for each file.

Enterprise results are identical to the previous version (706 techniques, 44 mitigations, 1477 links).

#### 1. Setting up
The libraries are imported and the data folder is located.

*Why:* the path is relative to the project folder (`../data`) instead of a fixed path on one person's computer (like `C:\Users\...`), so the notebook runs for every team member who has cloned the project. If the notebook is run from the project root, it falls back to `data/`.

In [1]:
import json
import pandas as pd
from pathlib import Path

data_dir = Path("../data")
if not data_dir.exists():
    data_dir = Path("data")

print("Data folder:", data_dir.resolve())

Data folder: C:\Users\noral\Documents\IntroToDataScience\Project\DAtabase 2\data


#### 2. Getting the ATT&CK files from the repository
The ATT&CK datasets are downloaded from the team's fork of MITRE's `attack-stix-data` repository on GitHub and stored in `data/raw/`.

*Why:* the data comes from one shared online source, so every team member uses exactly the same files and nobody has to download them by hand. A file is only downloaded if it is not already in `data/raw/`, so the large Enterprise file (about 55 MB) is not downloaded again on every run. To get a newer ATT&CK release, delete the file from `data/raw/` and run the cell again.

ATT&CK is split into three separate datasets (Enterprise, Mobile, ICS); keeping them in one list makes it easy to add or remove a domain later.

In [2]:
import requests

ATTACK_REPO = "https://raw.githubusercontent.com/Mastercoder-1000/attack-stix-data/master"
domains = ["enterprise-attack", "ics-attack"]   # "mobile-attack" not needed for the CISA scenarios

raw_dir = data_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

for d in domains:
    path = raw_dir / f"{d}.json"
    if not path.exists():
        url = f"{ATTACK_REPO}/{d}/{d}.json"
        print(f"Downloading {url} ...")
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)
    print(f"{d}: {path.stat().st_size / 1_000_000:.1f} MB")

enterprise-attack: 55.3 MB
ics-attack: 5.3 MB


#### 3. Helper functions
Small functions for extracting MITRE IDs and tactics and for filtering out outdated objects.

*Why:* the same logic is needed for every domain, so it is written once. `is_active` is a small safeguard: it treats missing values as "not revoked/deprecated" and does not crash with a `KeyError` if a future ATT&CK file (or a different domain) happens not to contain the `revoked` or `x_mitre_deprecated` columns. The current Enterprise and ICS files both contain them, so the result is the same as the previous filter.

In [3]:
def get_mitre_id(refs):
    """Returns the ATT&CK ID (e.g. T1566, M1031) from external_references."""
    if not isinstance(refs, list):
        return None
    for ref in refs:
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id")
    return None


def get_tactics(phases):
    """Returns the tactic names from kill_chain_phases as a comma-separated string."""
    if not isinstance(phases, list):
        return None
    return ", ".join(p["phase_name"] for p in phases)


def is_active(df):
    """True for objects that are not revoked or deprecated (works even if the columns are missing)."""
    revoked = df.get("revoked", pd.Series(False, index=df.index)).fillna(False)
    deprecated = df.get("x_mitre_deprecated", pd.Series(False, index=df.index)).fillna(False)
    return (revoked != True) & (deprecated != True)

#### 4. Extraction function
One function that extracts techniques, mitigations and their relationships from a single ATT&CK file.

*Why:* the steps are the same as before (select active techniques, select active mitigations, keep the "mitigates" relationships, join names onto IDs), but wrapping them in a function lets the same code run for Enterprise and ICS without copy-pasting. A `domain` column is added so the rows stay distinguishable after combining.

In [4]:
def extract_domain(domain):
    with open(raw_dir / f"{domain}.json", encoding="utf-8") as f:
        objects = pd.DataFrame(json.load(f)["objects"])

    # active attack techniques
    techniques = objects[(objects["type"] == "attack-pattern") & is_active(objects)].copy()
    techniques["technique_id"] = techniques["external_references"].apply(get_mitre_id)
    techniques["tactics"] = techniques["kill_chain_phases"].apply(get_tactics)
    techniques = techniques[["id", "technique_id", "name", "description", "tactics"]]

    # active mitigations
    mitigations = objects[(objects["type"] == "course-of-action") & is_active(objects)].copy()
    mitigations["mitigation_id"] = mitigations["external_references"].apply(get_mitre_id)
    mitigations = mitigations[["id", "mitigation_id", "name", "description"]]

    # "mitigates" relationships between active mitigations and active techniques
    rel = objects[
        (objects["type"] == "relationship")
        & (objects["relationship_type"] == "mitigates")
        & is_active(objects)
    ]
    rel = rel[rel["source_ref"].isin(mitigations["id"]) & rel["target_ref"].isin(techniques["id"])]
    rel = rel[["source_ref", "target_ref"]].drop_duplicates()

    # readable technique-mitigation table
    technique_mitigations = (
        rel.merge(mitigations[["id", "mitigation_id", "name"]],
                  left_on="source_ref", right_on="id", validate="many_to_one")
           .rename(columns={"name": "mitigation_name"}).drop(columns="id")
           .merge(techniques[["id", "technique_id", "name"]],
                  left_on="target_ref", right_on="id", validate="many_to_one")
           .rename(columns={"name": "technique_name"})
           [["technique_id", "technique_name", "mitigation_id", "mitigation_name"]]
    )

    for df in (techniques, mitigations, technique_mitigations):
        df.insert(0, "domain", domain)

    return techniques, mitigations, technique_mitigations

#### 5. Extracting all domains
The function is applied to each domain and the results are combined into three tables.

*Why:* one combined table per object type is easier to join with the CISA data later than separate tables per domain. The per-domain counts are printed so they can be compared with the previous version.

In [5]:
all_tech, all_mit, all_rel = [], [], []

for d in domains:
    t, m, r = extract_domain(d)
    all_tech.append(t)
    all_mit.append(m)
    all_rel.append(r)
    print(f"{d}: {len(t)} techniques, {len(m)} mitigations, {len(r)} technique-mitigation links")

techniques = pd.concat(all_tech, ignore_index=True)
mitigations = pd.concat(all_mit, ignore_index=True)
technique_mitigations = pd.concat(all_rel, ignore_index=True)

print("\nTotal:", len(techniques), "techniques,", len(mitigations), "mitigations,",
      len(technique_mitigations), "links")

enterprise-attack: 706 techniques, 44 mitigations, 1477 technique-mitigation links
ics-attack: 104 techniques, 53 mitigations, 424 technique-mitigation links

Total: 810 techniques, 97 mitigations, 1901 links


#### 6. Validating the data
The extracted IDs and relationships are checked for missing values and duplicates.

*Why:* duplicates are now checked **within each domain** (`domain` + ID) instead of across the whole table. Each ATT&CK domain is a separate dataset, so the correct rule is that an ID must be unique inside its own domain.

In [6]:
assert techniques["technique_id"].notna().all(), "Missing technique IDs"
assert mitigations["mitigation_id"].notna().all(), "Missing mitigation IDs"
assert not techniques.duplicated(["domain", "technique_id"]).any(), "Duplicate technique IDs"
assert not mitigations.duplicated(["domain", "mitigation_id"]).any(), "Duplicate mitigation IDs"
assert technique_mitigations.notna().all().all(), "Missing values in relationships"

print("All validation checks passed.")
display(techniques.groupby("domain").size().rename("techniques").to_frame()
        .join(mitigations.groupby("domain").size().rename("mitigations"))
        .join(technique_mitigations.groupby("domain").size().rename("links")))

All validation checks passed.


,techniques,mitigations,links
domain,,,
enterprise-attack,706,44,1477
ics-attack,104,53,424


#### 7. Examples
Mitigations for one Enterprise technique and one ICS technique.

*Why:* a quick sanity check that the relationships make sense. Phishing (T1566) is used in several CISA scenarios; Loss of Safety (T0880) is relevant for the Industrial Control Systems scenario.

In [7]:
display(technique_mitigations[technique_mitigations["technique_id"] == "T1566"])
display(technique_mitigations[technique_mitigations["technique_id"] == "T0880"])

,domain,technique_id,technique_name,mitigation_id,mitigation_name
98,enterprise-attack,T1566,Phishing,M1047,Audit
651,enterprise-attack,T1566,Phishing,M1031,Network Intrusion Prevention
681,enterprise-attack,T1566,Phishing,M1054,Software Configuration
936,enterprise-attack,T1566,Phishing,M1021,Restrict Web-Based Content
1353,enterprise-attack,T1566,Phishing,M1049,Antivirus/Antimalware
1365,enterprise-attack,T1566,Phishing,M1017,User Training


,domain,technique_id,technique_name,mitigation_id,mitigation_name
1505,ics-attack,T0880,Loss of Safety,M0812,Safety Instrumented Systems
1535,ics-attack,T0880,Loss of Safety,M0805,Mechanical Protection Layers


#### 8. Saving the datasets
The combined tables are saved for the CISA-MITRE mapping.

*Why:* the file names are the same as before, so the other notebooks keep working. The only difference in the files is the extra `domain` column.

In [8]:
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

techniques.to_csv(processed_dir / "mitre_techniques.csv", index=False, encoding="utf-8-sig")
mitigations.to_csv(processed_dir / "mitre_mitigations.csv", index=False, encoding="utf-8-sig")
technique_mitigations.to_csv(processed_dir / "technique_mitigations.csv", index=False, encoding="utf-8-sig")

print("MITRE datasets saved to", processed_dir)

MITRE datasets saved to data\processed
